# Week 2 companion notebook

Diffusion and score-based models, reproduced with NumPy and Matplotlib only (no deep-learning
framework needed). Every reverse process here uses the **exact** score of a mixture-of-Gaussians
target, so you can see the machinery work before training a network in Problems 3 and 4.

Sections mirror the [Week 2 notes](../../weeks/week2/notes.html):
1. The forward process and its perturbation kernel
2. The exact score, reverse SDE, and probability-flow ODE
3. Denoising score matching: the conditional mean is the score
4. The EDM design space: preconditioning
5. The score is a force field: the Mueller-Brown surface

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# A 'molecule-like' target: three Gaussians at the vertices of a triangle.
# The variance-exploding (EDM-style) perturbed marginals, score, and samples
# are all closed form, which is what makes everything below exact.
MODES = np.array([[0.0, 1.15], [-1.0, -0.6], [1.0, -0.6]])
WEIGHTS = np.array([1/3, 1/3, 1/3])
S0 = 0.16  # clean per-mode std

def mog_var(sigma):
    return S0**2 + sigma**2

def mog_score(P, sigma):
    var = mog_var(sigma)
    d = MODES[None, :, :] - P[:, None, :]          # (M, K, 2)
    logw = np.log(WEIGHTS) - (d**2).sum(-1) / (2*var)
    logw -= logw.max(1, keepdims=True)
    w = np.exp(logw); w /= w.sum(1, keepdims=True)
    return (w[..., None] * d).sum(1) / var

def mog_sample(n, sigma, rng):
    k = rng.choice(len(MODES), size=n, p=WEIGHTS)
    return MODES[k] + np.sqrt(mog_var(sigma)) * rng.standard_normal((n, 2))

## 1. The forward process and its perturbation kernel

The forward process convolves the data with a Gaussian of growing width,
$p_\sigma = p_\text{data} * \mathcal{N}(0, \sigma^2 I)$. As $\sigma$ grows the modes merge and the
distribution relaxes to a single wide Gaussian; the signal-to-noise ratio $\approx \sigma^{-2}$ falls.

In [ ]:
rng = np.random.default_rng(1)
sigs = [0.05, 0.3, 0.9, 2.5]
fig, axes = plt.subplots(1, 4, figsize=(13, 3.3))
for ax, s in zip(axes, sigs):
    pts = mog_sample(1500, s, rng)
    ax.scatter(pts[:, 0], pts[:, 1], s=5, color='#b3132a', alpha=0.45, edgecolors='none')
    ax.set_xlim(-3.4, 3.4); ax.set_ylim(-3.4, 3.4); ax.set_aspect('equal')
    ax.set_title(f'sigma = {s}   (SNR ~ {1/s**2:.2f})'); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## 2. Exact score, reverse SDE, and probability-flow ODE

Two samplers share the same trained model. The **probability-flow ODE** is deterministic,
$dx = -\sigma\, \nabla\log p_\sigma(x)\, d\sigma$ integrated from large to small $\sigma$; the
**reverse SDE** adds a Brownian term. Both reproduce the same marginals.

In [ ]:
def pf_ode_reverse(x, sigmas, score_fn):
    traj = [x.copy()]
    for i in range(len(sigmas) - 1):
        s, s_next = sigmas[i], sigmas[i+1]
        d = -s * score_fn(x, s)
        x_eu = x + (s_next - s) * d
        if s_next > 0:
            d2 = -s_next * score_fn(x_eu, s_next)
            x = x + (s_next - s) * 0.5 * (d + d2)   # Heun
        else:
            x = x_eu
        traj.append(x.copy())
    return np.array(traj)

def reverse_sde(x, sigmas, score_fn, rng, noise_scale=0.5):
    traj = [x.copy()]
    for i in range(len(sigmas) - 1):
        s, s_next = sigmas[i], sigmas[i+1]
        dsig = s_next - s
        x = x - 2*s*score_fn(x, s)*dsig
        if s_next > 0:
            x = x + noise_scale*np.sqrt(2*s*(-dsig))*rng.standard_normal(x.shape)
        traj.append(x.copy())
    return np.array(traj)

sigmas = np.concatenate([np.geomspace(3.0, 0.03, 60), [0.0]])
x0 = 3.0 * np.random.default_rng(7).standard_normal((6, 2))
ode = pf_ode_reverse(x0.copy(), sigmas, mog_score)
sde = reverse_sde(x0.copy(), sigmas, mog_score, np.random.default_rng(11))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4.6))
for ax, traj, col, ttl in [(a1, ode, '#2f6fb3', 'probability-flow ODE'),
                           (a2, sde, '#b3132a', 'reverse SDE')]:
    for j in range(x0.shape[0]):
        ax.plot(traj[:, j, 0], traj[:, j, 1], color=col, lw=1.0, alpha=0.8)
    ax.scatter(MODES[:, 0], MODES[:, 1], s=80, facecolors='none', edgecolors='#444', zorder=4)
    ax.set_title(ttl); ax.set_aspect('equal'); ax.set_xlim(-3.2, 3.2); ax.set_ylim(-3.2, 3.2)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## 3. Denoising score matching: the conditional mean is the score

We never observe $\nabla\log p_\sigma$ directly, but for each noisy sample $x_\sigma = x_0 + \sigma\epsilon$
the vector $(x_0 - x_\sigma)/\sigma^2$ is an unbiased, very noisy estimate of it. Averaging these
per-sample targets over all clean points that could have produced a given $x_\sigma$ recovers the
exact score. Below we check this by Monte Carlo against the closed form, no training required.

In [ ]:
rng = np.random.default_rng(0)
sigma = 0.6
query = np.array([[0.4, 0.5]])           # a single point x where we want the score

# Monte Carlo: sample clean x0 ~ data, weight by the kernel p(x | x0), average targets.
M = 400000
x0s = mog_sample(M, 0.0, rng)            # clean data samples
w = np.exp(-((query - x0s)**2).sum(1) / (2*sigma**2))   # Gaussian kernel weight
w /= w.sum()
target = (x0s - query) / sigma**2
mc_score = (w[:, None] * target).sum(0)

exact = mog_score(query, sigma)[0]
print('Monte-Carlo DSM score:', np.round(mc_score, 3))
print('closed-form score:    ', np.round(exact, 3))

## 4. The EDM design space: preconditioning

EDM wraps the raw network so its inputs and targets stay unit-scale at every noise level,
with coefficients fixed by $\sigma_\text{data}$. At small $\sigma$ the denoiser is nearly the identity;
at large $\sigma$ it must predict the clean signal from almost pure noise.

In [ ]:
sd = 0.5
sig = np.logspace(-3, 2, 400)
c_skip = sd**2 / (sig**2 + sd**2)
c_out  = sig*sd / np.sqrt(sig**2 + sd**2)
c_in   = 1.0 / np.sqrt(sig**2 + sd**2)

plt.figure(figsize=(6.2, 4))
plt.plot(sig, c_skip, color='#b3132a', lw=2, label='c_skip')
plt.plot(sig, c_out,  color='#2f6fb3', lw=2, label='c_out')
plt.plot(sig, c_in,   color='#1f9d6b', lw=2, label='c_in')
plt.axvline(sd, color='#888', ls='--', lw=1)
plt.xscale('log'); plt.xlabel('noise level sigma'); plt.ylabel('coefficient')
plt.title('EDM preconditioning'); plt.legend(frameon=False); plt.tight_layout(); plt.show()

## 5. The score is a force field: the Mueller-Brown surface

For Boltzmann data $p \propto e^{-V/k_BT}$ the score is the force $-\nabla V / k_BT$. The
Mueller-Brown potential is the classic chemistry test surface (three basins, one barrier) used in
Problem 3. Here is the surface and its Boltzmann distribution, the target a trained score model
must reproduce, weights and all.

In [ ]:
MB_A = np.array([-200., -100., -170., 15.]); MB_a = np.array([-1., -1., -6.5, 0.7])
MB_b = np.array([0., 0., 11., 0.6]);          MB_c = np.array([-10., -10., -6.5, 0.7])
MB_x0 = np.array([1., 0., -0.5, -1.]);        MB_y0 = np.array([0., 0.5, 1.5, 1.])

def muller_brown(X, Y):
    V = np.zeros_like(X)
    for k in range(4):
        dx, dy = X - MB_x0[k], Y - MB_y0[k]
        V += MB_A[k]*np.exp(MB_a[k]*dx**2 + MB_b[k]*dx*dy + MB_c[k]*dy**2)
    return V

xs = np.linspace(-1.6, 1.1, 300); ys = np.linspace(-0.4, 2.05, 300)
X, Y = np.meshgrid(xs, ys); V = muller_brown(X, Y)
kT = 25.0; p = np.exp(-(V - V.min())/kT); p /= p.sum()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.4))
a1.contourf(X, Y, np.clip(V, -150, 100), levels=30, cmap='terrain_r')
a1.set_title('Mueller-Brown potential V(x, y)')
rng = np.random.default_rng(2)
idx = rng.choice(p.size, size=4000, p=p.ravel())
iy, ix = np.unravel_index(idx, p.shape)
a2.hexbin(xs[ix], ys[iy], gridsize=46, cmap='Reds', extent=(-1.6, 1.1, -0.4, 2.05))
a2.set_title('Boltzmann samples  p ~ exp(-V/kT)')
for ax in (a1, a2): ax.set_xlabel('x'); ax.set_ylabel('y')
plt.tight_layout(); plt.show()

## Play with it

- Change `noise_scale` in `reverse_sde` and watch the stochastic paths get rougher or smoother.
- Lower `kT` for the Mueller-Brown surface and see the deepest basin swallow all the probability,
  the multimodality problem a generator has to solve.
- Move the `query` point in Section 3 into a low-density region and watch the Monte-Carlo score
  estimate get noisier, the reason NCSN uses a ladder of noise levels.
- Swap `MODES` for your own layout (a ring, a chain) and re-run the reverse samplers.